# 2 hidden class - GMM

## Import libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## GMM training

# Calcula CM para todos

In [3]:
filenames = ['0_2','0_3','0_4','0_5','2_3','2_4','2_5','3_4','3_5','4_5']

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

pd.set_option('future.no_silent_downcasting', True)

BIC score applied to decide the number of GMM components (1 -10)

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV
import seaborn as sns
import os

exp_train = []
exp_test = []
y_true_all_exp = []
cand_ths_all_exp = []

def gmm_bic_score(estimator, X):
    """Callable to pass to GridSearchCV that will use the BIC score."""
    # Make it negative since GridSearchCV expects a score to maximize
    return -estimator.bic(X)


param_grid = {
    "n_components": range(1, 11),
    "covariance_type": ["spherical", "tied", "diag", "full"],
    "random_state": [123],
}

for z in range(len(filenames)):
    train_encoded_df = pd.read_csv(f'train_encoded_{filenames[z]}_hidden.csv')
    test_encoded_df = pd.read_csv(f'test_encoded_{filenames[z]}_hidden.csv')
    print(filenames[z])
    hidden_classes = list(map(int, filenames[z].split('_'))) # Classes hidden from training
    for i in range(len(labels_str)):
        train_encoded_df['Label'] = train_encoded_df['Label'].replace(labels_str[i],i)
        test_encoded_df['Label'] = test_encoded_df['Label'].replace(labels_str[i],i)

    display(train_encoded_df['Label'].value_counts())
    if f'train_{filenames[z]}_GMM_BIC_1_10_scores.csv' not in os.listdir():
        
        gmms = []
        
        for i in range(len(labels_str)):
            if i not in hidden_classes:
                print(f'Tipo: {i} = {labels_str[i]}')
                grid_search = GridSearchCV(
                    GaussianMixture(), param_grid=param_grid, scoring=gmm_bic_score
                )
                grid_search.fit(train_encoded_df[train_encoded_df['Label'] == i].drop(columns=['Label']).to_numpy())
                print(f"Melhor modelo: n_componentes: {grid_search.best_params_['n_components']} covariance_type: {grid_search.best_params_['covariance_type']}")
                gmms.append(grid_search.best_estimator_)
                df = pd.DataFrame(grid_search.cv_results_)[
                    ["param_n_components", "param_covariance_type", "mean_test_score"]
                ]
                df["mean_test_score"] = -df["mean_test_score"]
                df = df.rename(
                    columns={
                        "param_n_components": "Number of components",
                        "param_covariance_type": "Type of covariance",
                        "mean_test_score": "BIC score",
                    }
                )
                df.sort_values(by="BIC score").head()
                sns.catplot(
                    data=df,
                    kind="bar",
                    x="Number of components",
                    y="BIC score",
                    hue="Type of covariance",
                    
                ).set(title = f"{labels_str[i]} n_components: {grid_search.best_params_['n_components']} covariance_type: {grid_search.best_params_['covariance_type']}")
                plt.show()
            else:
                gmms.append(None)
        
        scores = []
        for i, row in test_encoded_df.drop(columns=['Label']).iterrows():
            max_dist = -np.inf
            pred = -1
            scores.append([])
            for j in range(len(labels_str)):
                if j not in hidden_classes:
                    inside = False
                    max = -np.inf
                    score = gmms[j].score_samples(row.to_numpy().reshape(1, -1))[0]
                    scores[i].append(score)
                else:
                    scores[i].append(np.nan)
        
        
        display(pd.DataFrame(scores))
        exp_test.append(scores)
        pd.DataFrame(scores).to_csv(f'test_{filenames[z]}_GMM_BIC_1_10_scores.csv', index=False)